# Geometric Organization of Carbon Allotropes

## Setup

In [ ]:
from setup import *

import matplotlib.patches as mpatches

## Compute Carbon Projections

In [ ]:
size_cmap = plt.get_cmap("Spectral")
group_colormaps = {"Amorphous": plt.get_cmap("Reds"),
               "Crystalline": plt.get_cmap("YlOrRd"),
               "Layered_sp2": plt.get_cmap("Greens"),
               "Ordered_allotropes": plt.get_cmap("Blues"),
               "Liquid": plt.get_cmap("Purples"),
               "Surfaces": plt.get_cmap("YlGnBu"),
               "Defects": plt.get_cmap("YlGnBu"),
               "Small": plt.get_cmap("Greys"),
               "Other": plt.get_cmap("Reds")}
category_groups = {"Amorphous_Bulk":("Amorphous",0.5),"Amorphous_Surfaces":("Amorphous",0.625),"LD_Iter1":("Amorphous",0.75),
                "Crystalline_Bulk":("Crystalline",0.25),"Crystalline_RSS":("Crystalline",0.375),"SACADA":("Crystalline",0.5),
                "Graphene":("Layered_sp2",0.75),"Graphite":("Layered_sp2",0.825),"Graphite_Layer_Sep":("Layered_sp2",1.0),
                "Diamond":("Ordered_allotropes",0.75),"Fullerenes":("Ordered_allotropes",0.825),"Nanotubes":("Ordered_allotropes",1.0),
                "Liquid":("Liquid",0.375),"Liquid_Interface":("Liquid",0.5),
                "Surfaces":("Surfaces",0.4),
                "Defects":("Defects",0.6),
                "Dimer":("Small",0.25),"Single_Atom":("Small",0.375)}
category_labels = {
    "Amorphous_Bulk": "Amorphous Bulk",
    "Amorphous_Surfaces": "Amorphous Surfaces",
    "LD_Iter1": "Liquid-Deposited",
    "Crystalline_Bulk": "Crystalline Bulk",
    "Crystalline_RSS": "Crystalline",
    "SACADA": "SACADA Allotropes",
    "Graphene": "Graphene",
    "Graphite": "Graphite",
    "Graphite_Layer_Sep": "Graphite Layer",
    "Diamond": "Diamond",
    "Fullerenes": "Fullerenes",
    "Nanotubes": "Nanotubes",
    "Liquid": "Liquid Carbon",
    "Liquid_Interface": "Liquid Interface",
    "Surfaces": "Surfaces",
    "Defects": "Defects",
    "Dimer": "Dimer",
    "Single_Atom": "Single Atom",
}

carbon_results = {}
for variant, directory in CARBON_DIRS.items():
    dataset = ConformerEdgeDataset(str(directory), num_conformers=1, sample_per_batch=1)
    loader = DataLoader(dataset, batch_size=4, collate_fn=ConformerEdge_collate_fn, shuffle=False, pin_memory=True)

    # point colors depend only on the dataset (not the model), so build them once and reuse for every model
    category_colors = []
    size_colors = []
    for file, data in zip(dataset.paths, dataset.data):
        num_molecules = data["charge"].shape[0]
        words = Path(file).stem.split("_")
        carbon_name = "_".join(words[:-1])
        num_atoms = int(words[-1][1:])
        group, color_value = category_groups[carbon_name]
        category_colors.extend([group_colormaps[group](color_value)] * num_molecules)
        size_colors.extend(list(size_cmap([min(300, num_atoms) / 300.0] * num_molecules)))
    projections = {}
    for name in ("nopre", "NT", "ANT"):
        model = EncoderModel.load_model(CHECKPOINTS[name], device=DEVICE).eval()
        embeddings = encode_batches(model, tqdm(loader, desc="Embedding carbon structures")).numpy()
        assert len(embeddings) == len(category_colors)
        projections[name] = compute_tsne(embeddings, perplexity=100, seed=0)
    carbon_results[variant] = {
        "projections": projections,
        "category_colors": category_colors,
        "size_colors": size_colors,
    }

## Figure 7: Carbon Allotrope Organization

In [ ]:
panel_size = (5 * cm, 5 * cm)

def gap20_scatter(tsne_data, colors, out_path):
    """Square t-SNE scatter inside a thin box with a 0.05 figure-fraction margin (no ticks, tick labels, title or legend)."""
    fig = plt.figure(figsize=panel_size)
    ax = fig.add_axes([0.01, 0.01, 0.98, 0.98])
    ax.scatter(tsne_data[:, 0], tsne_data[:, 1], c=colors, s=0.7, edgecolors='none')
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():  # all four sides visible
        spine.set_visible(True)
        spine.set_linewidth(0.25)
        spine.set_edgecolor("black")
    fig.savefig(out_path, dpi=600)
    plt.show()

def plot_carbon_panel(variant, model_name, coloring):
    data = carbon_results[variant]
    suffix = "" if variant == "unit_cell" else f"_{variant}"
    file_label = "categories" if coloring == "category" else "MW"
    gap20_scatter(data["projections"][model_name], data[f"{coloring}_colors"],
                  OUTPUT_DIR / f"GAP20{suffix}_{file_label}_{model_name}.png")
    
plot_carbon_panel('unit_cell', 'nopre', 'category')
plot_carbon_panel('unit_cell', 'nopre', 'size')
plot_carbon_panel('unit_cell', 'NT', 'category')
plot_carbon_panel('unit_cell', 'NT', 'size')
plot_carbon_panel('unit_cell', 'ANT', 'category')
plot_carbon_panel('unit_cell', 'ANT', 'size')

def gap20_categories_legend(out_path, ncol=1, width_cm=1.92, height_cm=4, max_fontsize=5, frame=False):
    """Category legend laid out by hand on an exact width_cm x height_cm figure: ncol columns (filled top to bottom,
    so each group stays together), rows evenly spread over the full height and aligned across columns.
    The font is the largest size <= max_fontsize (0.25 pt steps) at which every label fits its column; a label is
    wrapped onto two lines (after its first hyphen or space) only when it doesn't fit on one."""
    pad, col_gap, swatch_w, swatch_h, text_gap = 0.03, 0.04, 0.12, 0.09, 0.03  # cm
    entries = [(category_labels[name], group_colormaps[group](color_value)) for name, (group, color_value) in category_groups.items()]
    nrow = math.ceil(len(entries) / ncol)
    col_w = (width_cm - 2 * pad - (ncol - 1) * col_gap) / ncol
    row_h = (height_cm - 2 * pad) / nrow
    text_w = col_w - swatch_w - text_gap  # room left for the label in each column
    fig = plt.figure(figsize=(width_cm * cm, height_cm * cm), dpi=600)  # measure text at the dpi it's saved at
    renderer = fig.canvas.get_renderer()
    to_fig = lambda x, y: (x / width_cm, y / height_cm)  # cm -> figure fraction

    def text_width_cm(s, fontsize):
        t = fig.text(0, 0, s, fontsize=fontsize, linespacing=1.0)
        w = t.get_window_extent(renderer).width / fig.dpi * 2.54
        t.remove()
        return w

    def fit_label(label, fontsize):
        """The label as it should be drawn (one line, else wrapped to two), or None if neither fits."""
        wrapped = label.replace("-", "-\n", 1) if "-" in label else label.replace(" ", "\n", 1)
        return next((s for s in (label, wrapped) if text_width_cm(s, fontsize) <= text_w), None)

    fontsize = max_fontsize
    while True:
        fitted = [fit_label(label, fontsize) for label, _ in entries]
        if None not in fitted or fontsize <= 3:
            break
        fontsize -= 0.25
    print(f"categories legend: {fontsize} pt font, {ncol} columns, {width_cm} x {height_cm} cm")
    for label, s in zip([label for label, _ in entries], fitted):
        if s is None:
            print(f"warning: legend label {label!r} is wider than its column even at {fontsize} pt; raise width_cm")

    for k, ((label, color), s) in enumerate(zip(entries, fitted)):
        col, row = divmod(k, nrow)
        x = pad + col * (col_w + col_gap)
        y = height_cm - pad - (row + 0.5) * row_h  # row center
        fig.add_artist(mpatches.Rectangle(to_fig(x, y - swatch_h / 2), swatch_w / width_cm, swatch_h / height_cm,
                                          transform=fig.transFigure, facecolor=color, edgecolor="none"))
        fig.text(*to_fig(x + swatch_w + text_gap, y), s or label, fontsize=fontsize, va="center", ha="left", linespacing=1.0)

    if frame:  # thin light frame like the default legend box, inset so the line isn't clipped at the edge
        inset = 0.01
        fig.add_artist(mpatches.Rectangle(to_fig(inset, inset), 1 - 2 * inset / width_cm, 1 - 2 * inset / height_cm,
                                          transform=fig.transFigure, fill=False, edgecolor="0.8", linewidth=0.3))
    fig.savefig(out_path, dpi=600)  # no bbox_inches="tight": keep the exact width_cm x height_cm size
    plt.show()

gap20_categories_legend(OUTPUT_DIR / "GAP20_categories_legend.png")
cbar_height_cm = 3.0
fig = plt.figure(figsize=panel_size)
cax = fig.add_axes([0.1, 0.5 - cbar_height_cm / 10, 0.06, cbar_height_cm / 5])  # 0.3 cm wide, centered vertically
norm = matplotlib.colors.Normalize(vmin=0, vmax=300)
sm = matplotlib.cm.ScalarMappable(cmap=size_cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label("Carbon Count", fontsize=8)
cbar.ax.tick_params(labelsize=6, width=0.25)
ticks = cbar.get_ticks()
tick_labels = [f"{int(t)}" for t in ticks]
if len(tick_labels) > 0:
    tick_labels[0] = f"{int(ticks[0])}"
    tick_labels[-1] = f"≥{int(ticks[-1])}"
cbar.set_ticks(ticks)
cbar.set_ticklabels(tick_labels)
cbar.outline.set_linewidth(0.25)
fig.savefig(OUTPUT_DIR / "GAP20_MW_legend.png", dpi=600, bbox_inches="tight", pad_inches=0.01)
plt.show()

## Figure S10: Carbon Environments Within 5 Å

In [ ]:
plot_carbon_panel('periodic_5A', 'nopre', 'category')
plot_carbon_panel('periodic_5A', 'nopre', 'size')
plot_carbon_panel('periodic_5A', 'NT', 'category')
plot_carbon_panel('periodic_5A', 'NT', 'size')
plot_carbon_panel('periodic_5A', 'ANT', 'category')
plot_carbon_panel('periodic_5A', 'ANT', 'size')

## Figure S11: Carbon Environments Within 8 Å

In [ ]:
plot_carbon_panel('periodic_8A', 'nopre', 'category')
plot_carbon_panel('periodic_8A', 'nopre', 'size')
plot_carbon_panel('periodic_8A', 'NT', 'category')
plot_carbon_panel('periodic_8A', 'NT', 'size')
plot_carbon_panel('periodic_8A', 'ANT', 'category')
plot_carbon_panel('periodic_8A', 'ANT', 'size')